<a href="https://colab.research.google.com/github/raulmabcn/DataAnalyst/blob/main/Foundations/Python/Exercices/Bloque3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



5️⃣ Funciones analíticas

Crea una función que reciba una comunidad autónoma y devuelva:

Hospital con mayor ocupación

Hospital con menor satisfacción

Diferencia entre el máximo y mínimo índice de satisfacción

Total de pacientes asociados a esa comunidad

Crea una función que detecte “hospitales en riesgo”, definidos como:

ocupación > 80%

índice_satisfacción < media nacional

carga_trabajo_médico superior al percentil 75



In [2]:
import pandas as pd
data = pd.read_csv('hospitals.csv', sep=';', encoding='latin-1')
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 170 entries, 0 to 169
Data columns (total 10 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   hospital_id                 170 non-null    int64  
 1   nombre                      170 non-null    object 
 2   localidad                   170 non-null    object 
 3   provincia                   170 non-null    object 
 4   comunidad_autonoma          170 non-null    object 
 5   presupuesto_anual_millones  170 non-null    float64
 6   num_camas                   170 non-null    int64  
 7   num_medicos                 170 non-null    int64  
 8   indice_satisfaccion         170 non-null    float64
 9   porcentaje_ocupacion        170 non-null    float64
dtypes: float64(3), int64(3), object(4)
memory usage: 13.4+ KB


In [18]:
def load_patients_data():
   data_p = pd.read_csv( 'pacientes.csv', sep=';', encoding='latin-1')
   return data_p

def hospital_max_occupation( c_data ):
  return c_data.loc[c_data['porcentaje_ocupacion'].idxmax()][['nombre','porcentaje_ocupacion']]

def hospital_min_satisfaction( c_data ):
   return c_data.loc[c_data['indice_satisfaccion'].idxmin()][['nombre','indice_satisfaccion']]

def range_satisfaccion( c_data ):
    return c_data['indice_satisfaccion'].agg( rango=lambda x: x.max() - x.min() )

def total_patients( c_data ):
    data_p = load_patients_data()
    data_p_c = data_p.groupby('hospital_id', as_index=False).agg( num_patients = ( 'paciente_id', 'count') )
    c_data = c_data.merge( data_p_c, on='hospital_id', how ='left' )

    return c_data['num_patients'].sum()

def get_risks_hospitals( c_data ):

    national_mean = get_national_mean()

    percentage_condition = c_data['porcentaje_ocupacion'] > 80
    satisfaction_condition = national_mean > c_data['indice_satisfaccion']
    work_load_condition = (( c_data['num_camas'] / c_data['num_medicos'] ) * c_data['porcentaje_ocupacion']) > 75

    return c_data[percentage_condition & satisfaction_condition & work_load_condition]['nombre']


def get_national_mean():
  return data['indice_satisfaccion'].mean().round(2)

def community_analisys( c_data ):

    print( hospital_max_occupation(c_data) )
    print( hospital_min_satisfaction(c_data) )
    print( range_satisfaccion(c_data) )
    print( 'Total pacientes: ',total_patients( c_data ))
    print( get_risks_hospitals( c_data ) )


def get_community( ):
  list_communities = data['comunidad_autonoma'].unique().tolist()
  community = 'Andalucia'
  #community = input( 'Introduzca el nombre de la comunidad a analizar: ' )
  return community

def get_data_community( community ):
  return data[ data['comunidad_autonoma'] == community ]



community = get_community()
community_data = get_data_community( community )
community_analisys( community_data )





nombre                  Hospital Infanta Margarita
porcentaje_ocupacion                          99.7
Name: 5, dtype: object
nombre                 Hospital Universitario Virgen de la Victoria
indice_satisfaccion                                             1.6
Name: 13, dtype: object
rango    8.2
Name: indice_satisfaccion, dtype: float64
Total pacientes:  939
5                 Hospital Infanta Margarita
15    Hospital Universitario Virgen Macarena
18                    Hospital de San Carlos
23                      Hospital San Agustin
26                     Hospital de Antequera
Name: nombre, dtype: object


6️⃣ Análisis combinado multi-tabla

Calcula el ranking de hospitales por:

score = (indice_satisfaccion * 0.4)
        + (1 - porcentaje_ocupacion) * 0.3
        + (num_camas normalizado) * 0.3

Devuelve el Top 10.

Determina si existe correlación entre:

número de camas

índice de satisfacción

porcentaje de ocupación

Interpreta el resultado.

In [34]:
def get_hospitals_ranking( c_data, ranking = 10 ):

  n_beds_norm = c_data['num_camas'] - c_data['num_camas'].min()/ c_data['num_camas'].max() - c_data['num_camas'].min()

  score = ((c_data['indice_satisfaccion']/10)* 0.4) + (1 - (c_data['porcentaje_ocupacion']/100))* 0.3 + n_beds_norm* 0.3

  h_r_data = c_data.loc[ score.sort_values(ascending = False).index].head(ranking)

  return h_r_data


def analice_correlation( c_data, columns = [] ):

  print( c_data[columns].corr() )
  pass



data_hospitals = get_hospitals_ranking( community_data )
columns = ['num_camas', 'indice_satisfaccion', 'porcentaje_ocupacion']
analice_correlation( data_hospitals, columns )



                      num_camas  indice_satisfaccion  porcentaje_ocupacion
num_camas              1.000000            -0.216654              0.344284
indice_satisfaccion   -0.216654             1.000000              0.162839
porcentaje_ocupacion   0.344284             0.162839              1.000000


In [41]:
community_data.corr(numeric_only=True)['indice_satisfaccion'].drop('indice_satisfaccion').sort_values(key=abs, ascending=False)
#community_data.corrwith(community_data['indice_satisfaccion'], numeric_only=True )

,0
hospital_id,0.265382
presupuesto_anual_millones,-0.054840
num_camas,0.381979
num_medicos,0.214947
indice_satisfaccion,1.000000
porcentaje_ocupacion,-0.039000
